# Role A (OCR): first session

Needs the Colab secret `GITHUB_TOKEN` with **Notebook access** switched on for this notebook, and a GPU (Runtime > Change runtime type > GPU).

This notebook only checks that the data is ready and shows a few samples. Your own work goes into `src/ocr/` and into notebooks named `ocr_*`. Colab cannot push to GitHub: download your notebook (File > Download .ipynb), put it in the `notebooks/` folder of your clone and push from your laptop.

**First:** choose File > Save a copy in Drive and work in the copy, so the notebook in the shared folder stays clean for everybody. In the copy, switch on Notebook access for `GITHUB_TOKEN` again.


## 1. Connect to Drive and GitHub

In [ ]:
from google.colab import drive, userdata

drive.mount("/content/drive")

BRANCH = "main"  # use your own branch (a/ocr, b/corrector, c/eval) to test code you have pushed

token = userdata.get("GITHUB_TOKEN")
!rm -rf /content/repo
!git clone -q https://{token}@github.com/wickyhien18/Vietnamese-Handwriting-OCR.git /content/repo
!git -C /content/repo checkout -q {BRANCH}
%cd /content/repo
!pip install -q -r requirements.txt

import sys
sys.path.insert(0, "/content/repo")

DRIVE_ROOT = "/content/drive/MyDrive/Vietnamese-Handwriting-OCR"
DATA_DIR = f"{DRIVE_ROOT}/data"
CKPT_DIR = f"{DRIVE_ROOT}/checkpoints"
OUT_DIR = f"{DRIVE_ROOT}/outputs"


## 2. Copy the images to the Colab disk and extract them

Reading thousands of small files from Drive is slow, so every session copies the zip to the Colab disk first.


In [ ]:
import glob
import os
import shutil
import zipfile

UIT_DIR = f"{DATA_DIR}/uit_hwdb"
LOCAL_ROOT = "/content/uit_hwdb"
if not os.path.isdir(LOCAL_ROOT):
    archive_path = sorted(glob.glob(f"{UIT_DIR}/*.zip"))[0]
    shutil.copy(archive_path, "/content/uit_hwdb.zip")
    with zipfile.ZipFile("/content/uit_hwdb.zip") as archive:
        archive.extractall(LOCAL_ROOT)
print("images ready in", LOCAL_ROOT)


## 3. Load the index and the splits

In [ ]:
from src.common.index_io import read_index, read_splits, select_split

index = read_index(f"{OUT_DIR}/dataset_index.jsonl")
splits = read_splits(f"{OUT_DIR}/splits.json")
for name in ("train", "val", "test"):
    print(name, len(select_split(index, splits, name)))

missing = [r["image_path"] for r in index[:50] if not os.path.exists(os.path.join(LOCAL_ROOT, r["image_path"]))]
print("missing among the first 50 images:", missing)


## 4. Look at three samples (check that each image matches its text)

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

samples = select_split(index, splits, "train")[:3]
fig, axes = plt.subplots(len(samples), 1, figsize=(10, 3 * len(samples)))
for ax, rec in zip(axes, samples):
    ax.imshow(Image.open(os.path.join(LOCAL_ROOT, rec["image_path"])), cmap="gray")
    ax.set_title(rec["gt_text"])
    ax.axis("off")
plt.show()


## 5. Check the GPU

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())


## What to do next

1. Look at image sizes (height and width of 20 lines) and decide the fixed height for preprocessing.
2. Build the preprocessing and a small training loop, and make the model **overfit 50 training lines first**. If the loss does not go close to zero, the pipeline has a bug: find it before training on everything.
3. Save a checkpoint to `CKPT_DIR/ocr` after **every epoch** (Colab sessions get cut).
4. Measure validation CER with `corpus_error_rate` from `src/eval/align.py`.
5. Export `ocr_out.jsonl` (the contract fields, `image_path` as in the index) into `OUT_DIR` and check it with `python scripts/validate_ocr_out.py`.
